In [81]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os

from sklearn.feature_extraction.text import CountVectorizer
from sklearn.metrics.pairwise import cosine_similarity

from nltk.stem.porter import PorterStemmer

import pickle

pd.set_option('display.max_columns', 999)

pd.options.display.float_format = '{:20.2f}'.format

In [82]:
data = pd.read_csv("./data/Coursera.csv")

data.head()

,Course Name,University,Difficulty Level,Course Rating,Course URL,Course Description,Skills
0,Write A Feature Length Screenplay For Film Or ...,Michigan State University,Beginner,4.8,https://www.coursera.org/learn/write-a-feature...,Write a Full Length Feature Film Script In th...,Drama Comedy peering screenwriting film D...
1,Business Strategy: Business Model Canvas Analy...,Coursera Project Network,Beginner,4.8,https://www.coursera.org/learn/canvas-analysis...,"By the end of this guided project, you will be...",Finance business plan persona (user experien...
2,Silicon Thin Film Solar Cells,�cole Polytechnique,Advanced,4.1,https://www.coursera.org/learn/silicon-thin-fi...,This course consists of a general presentation...,chemistry physics Solar Energy film lambda...
3,Finance for Managers,IESE Business School,Intermediate,4.8,https://www.coursera.org/learn/operational-fin...,"When it comes to numbers, there is always more...",accounts receivable dupont analysis analysis...
4,Retrieve Data using Single-Table SQL Queries,Coursera Project Network,Beginner,4.6,https://www.coursera.org/learn/single-table-sq...,In this course you�ll learn how to effectively...,Data Analysis select (sql) database manageme...


In [83]:
data.shape

(3522, 7)

In [84]:
data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3522 entries, 0 to 3521
Data columns (total 7 columns):
 #   Column              Non-Null Count  Dtype 
---  ------              --------------  ----- 
 0   Course Name         3522 non-null   object
 1   University          3522 non-null   object
 2   Difficulty Level    3522 non-null   object
 3   Course Rating       3522 non-null   object
 4   Course URL          3522 non-null   object
 5   Course Description  3522 non-null   object
 6   Skills              3522 non-null   object
dtypes: object(7)
memory usage: 192.7+ KB


In [85]:
data.columns

Index(['Course Name', 'University', 'Difficulty Level', 'Course Rating',
       'Course URL', 'Course Description', 'Skills'],
      dtype='object')

In [86]:
data['Difficulty Level'].value_counts()

Difficulty Level
Beginner          1444
Advanced          1005
Intermediate       837
Conversant         186
Not Calibrated      50
Name: count, dtype: int64

In [87]:
data["Course Rating"].value_counts().nunique()

22

In [88]:
data['University'].value_counts().nunique()

50

In [89]:
data = data[['Course Name', 'Difficulty Level', 'Course URL', 'Course Description', 'Skills']]

data.head()

,Course Name,Difficulty Level,Course URL,Course Description,Skills
0,Write A Feature Length Screenplay For Film Or ...,Beginner,https://www.coursera.org/learn/write-a-feature...,Write a Full Length Feature Film Script In th...,Drama Comedy peering screenwriting film D...
1,Business Strategy: Business Model Canvas Analy...,Beginner,https://www.coursera.org/learn/canvas-analysis...,"By the end of this guided project, you will be...",Finance business plan persona (user experien...
2,Silicon Thin Film Solar Cells,Advanced,https://www.coursera.org/learn/silicon-thin-fi...,This course consists of a general presentation...,chemistry physics Solar Energy film lambda...
3,Finance for Managers,Intermediate,https://www.coursera.org/learn/operational-fin...,"When it comes to numbers, there is always more...",accounts receivable dupont analysis analysis...
4,Retrieve Data using Single-Table SQL Queries,Beginner,https://www.coursera.org/learn/single-table-sq...,In this course you�ll learn how to effectively...,Data Analysis select (sql) database manageme...


In [90]:
data['Course Name'] = data['Course Name'].str.replace(' ', ',')
data['Course Name'] = data['Course Name'].str.replace(':', '')
data['Course Name'] = data['Course Name'].str.replace(',,', ',')

data['Course Description'] = data['Course Description'].str.replace(' ', ',')
data['Course Description'] = data['Course Description'].str.replace(',,', ',')
data['Course Description'] = data['Course Description'].str.replace(':', '')
data['Course Description'] = data['Course Description'].str.replace('_', '')
data['Course Description'] = data['Course Description'].str.replace('(', '')
data['Course Description'] = data['Course Description'].str.replace(')', '')

data['Skills'] = data['Skills'].str.replace('(', '')
data['Skills'] = data['Skills'].str.replace(')', '')

In [91]:
data.head()

,Course Name,Difficulty Level,Course URL,Course Description,Skills
0,"Write,A,Feature,Length,Screenplay,For,Film,Or,...",Beginner,https://www.coursera.org/learn/write-a-feature...,"Write,a,Full,Length,Feature,Film,Script,In,thi...",Drama Comedy peering screenwriting film D...
1,"Business,Strategy,Business,Model,Canvas,Analys...",Beginner,https://www.coursera.org/learn/canvas-analysis...,"By,the,end,of,this,guided,project,you,will,be,...",Finance business plan persona user experienc...
2,"Silicon,Thin,Film,Solar,Cells",Advanced,https://www.coursera.org/learn/silicon-thin-fi...,"This,course,consists,of,a,general,presentation...",chemistry physics Solar Energy film lambda...
3,"Finance,for,Managers",Intermediate,https://www.coursera.org/learn/operational-fin...,"When,it,comes,to,numbers,there,is,always,more,...",accounts receivable dupont analysis analysis...
4,"Retrieve,Data,using,Single-Table,SQL,Queries",Beginner,https://www.coursera.org/learn/single-table-sq...,"In,this,course,you�ll,learn,how,to,effectively...",Data Analysis select sql database management...


In [92]:
data['tags'] = data['Course Name'] + data['Difficulty Level'] + data['Course Description'] + data['Skills']

In [93]:
data['tags'].iloc[0]

'Write,A,Feature,Length,Screenplay,For,Film,Or,TelevisionBeginnerWrite,a,Full,Length,Feature,Film,Script,In,this,course,you,will,write,a,complete,feature-length,screenplay,for,film,or,television,be,it,a,serious,drama,or,romantic,comedy,or,anything,in,between.,You�ll,learn,to,break,down,the,creative,process,into,components,and,you�ll,discover,a,structured,process,that,allows,you,to,produce,a,polished,and,pitch-ready,script,by,the,end,of,the,course.,Completing,this,project,will,increase,your,confidence,in,your,ideas,and,abilities,and,you�ll,feel,prepared,to,pitch,your,first,script,and,get,started,on,your,next.,This,is,a,course,designed,to,tap,into,your,creativity,and,is,based,in,"Active,Learning".,Most,of,the,actual,learning,takes,place,within,your,own,activities,-,that,is,writing!,You,will,learn,by,doing.,Here,is,a,link,to,a,TRAILER,for,the,course.,To,view,the,trailer,please,copy,and,paste,the,link,into,your,browser.,https//vimeo.com/382067900/b78b800dc0,Learner,review,"Love,the,approac

In [94]:
df = data[["Course Name", "Course URL", "tags"]]

df.head()

,Course Name,Course URL,tags
0,"Write,A,Feature,Length,Screenplay,For,Film,Or,...",https://www.coursera.org/learn/write-a-feature...,"Write,A,Feature,Length,Screenplay,For,Film,Or,..."
1,"Business,Strategy,Business,Model,Canvas,Analys...",https://www.coursera.org/learn/canvas-analysis...,"Business,Strategy,Business,Model,Canvas,Analys..."
2,"Silicon,Thin,Film,Solar,Cells",https://www.coursera.org/learn/silicon-thin-fi...,"Silicon,Thin,Film,Solar,CellsAdvancedThis,cour..."
3,"Finance,for,Managers",https://www.coursera.org/learn/operational-fin...,"Finance,for,ManagersIntermediateWhen,it,comes,..."
4,"Retrieve,Data,using,Single-Table,SQL,Queries",https://www.coursera.org/learn/single-table-sq...,"Retrieve,Data,using,Single-Table,SQL,QueriesBe..."


In [95]:
df["Course Name"] = df["Course Name"].str.replace(",", " ")

C:\Users\mshaf\AppData\Local\Temp\ipykernel_4908\3103243787.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df["Course Name"] = df["Course Name"].str.replace(",", " ")


In [96]:
df.rename(columns={"Course Name": "course_name", "Course URL": "course_url"})

,course_name,course_url,tags
0,Write A Feature Length Screenplay For Film Or ...,https://www.coursera.org/learn/write-a-feature...,"Write,A,Feature,Length,Screenplay,For,Film,Or,..."
1,Business Strategy Business Model Canvas Analys...,https://www.coursera.org/learn/canvas-analysis...,"Business,Strategy,Business,Model,Canvas,Analys..."
2,Silicon Thin Film Solar Cells,https://www.coursera.org/learn/silicon-thin-fi...,"Silicon,Thin,Film,Solar,CellsAdvancedThis,cour..."
3,Finance for Managers,https://www.coursera.org/learn/operational-fin...,"Finance,for,ManagersIntermediateWhen,it,comes,..."
4,Retrieve Data using Single-Table SQL Queries,https://www.coursera.org/learn/single-table-sq...,"Retrieve,Data,using,Single-Table,SQL,QueriesBe..."
...,...,...,...
3517,Capstone Retrieving Processing and Visualizing...,https://www.coursera.org/learn/python-data-vis...,"Capstone,Retrieving,Processing,and,Visualizing..."
3518,Patrick Henry Forgotten Founder,https://www.coursera.org/learn/henry,"Patrick,Henry,Forgotten,FounderIntermediate�Gi..."
3519,Business intelligence and data analytics Gener...,https://www.coursera.org/learn/business-intell...,"Business,intelligence,and,data,analytics,Gener..."
3520,Rigid Body Dynamics,https://www.coursera.org/learn/rigid-body-dyna...,"Rigid,Body,DynamicsBeginnerThis,course,teaches..."


In [97]:
df['tags'] = df['tags'].apply(lambda x: x.lower())

C:\Users\mshaf\AppData\Local\Temp\ipykernel_4908\4276611149.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['tags'] = df['tags'].apply(lambda x: x.lower())


In [98]:
df.shape

(3522, 3)

In [99]:
df.head()

,Course Name,Course URL,tags
0,Write A Feature Length Screenplay For Film Or ...,https://www.coursera.org/learn/write-a-feature...,"write,a,feature,length,screenplay,for,film,or,..."
1,Business Strategy Business Model Canvas Analys...,https://www.coursera.org/learn/canvas-analysis...,"business,strategy,business,model,canvas,analys..."
2,Silicon Thin Film Solar Cells,https://www.coursera.org/learn/silicon-thin-fi...,"silicon,thin,film,solar,cellsadvancedthis,cour..."
3,Finance for Managers,https://www.coursera.org/learn/operational-fin...,"finance,for,managersintermediatewhen,it,comes,..."
4,Retrieve Data using Single-Table SQL Queries,https://www.coursera.org/learn/single-table-sq...,"retrieve,data,using,single-table,sql,queriesbe..."


In [100]:
# initializing the CountVectorizer
cv = CountVectorizer(max_features=5000, stop_words= 'english')

In [101]:
# convert the tags into vectors
vectors = cv.fit_transform(df['tags']).toarray()

In [102]:
vectors[0]

array([0, 0, 0, ..., 0, 0, 0], shape=(5000,))

In [103]:
# initializing PorterStemmer
ps = PorterStemmer()

In [104]:
def stem(text):
    y = []
    for i in text.split():
        y.append(ps.stem(i))
        
    return " ".join(y)

In [105]:
stem("it's actually annoying")

"it' actual annoy"

In [106]:
stem("come get this thing")

'come get thi thing'

In [107]:
df['tags'] = df['tags'].apply(stem)

C:\Users\mshaf\AppData\Local\Temp\ipykernel_4908\866399325.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['tags'] = df['tags'].apply(stem)


In [108]:
df['tags'][0]

'write,a,feature,length,screenplay,for,film,or,televisionbeginnerwrite,a,full,length,feature,film,script,in,this,course,you,will,write,a,complete,feature-length,screenplay,for,film,or,television,be,it,a,serious,drama,or,romantic,comedy,or,anything,in,between.,you�ll,learn,to,break,down,the,creative,process,into,components,and,you�ll,discover,a,structured,process,that,allows,you,to,produce,a,polished,and,pitch-ready,script,by,the,end,of,the,course.,completing,this,project,will,increase,your,confidence,in,your,ideas,and,abilities,and,you�ll,feel,prepared,to,pitch,your,first,script,and,get,started,on,your,next.,this,is,a,course,designed,to,tap,into,your,creativity,and,is,based,in,"active,learning".,most,of,the,actual,learning,takes,place,within,your,own,activities,-,that,is,writing!,you,will,learn,by,doing.,here,is,a,link,to,a,trailer,for,the,course.,to,view,the,trailer,please,copy,and,paste,the,link,into,your,browser.,https//vimeo.com/382067900/b78b800dc0,learner,review,"love,the,approac

     Cosine Similarity for vectors 

In [109]:
similarity = cosine_similarity(vectors)

## Recommendation Engine

In [110]:
def recommend(course):
    course_index = df[df["Course Name"] == course].index[0]
    distance = similarity[course_index]
    course_list = sorted(list(enumerate(distance)), reverse=True, key=lambda x: x[1])[1:7]
    
    recommended_courses = []
    for i in course_list:
        course_name = df.iloc[i[0]]["Course Name"]
        course_url = df.iloc[i[0]]["Course URL"]
        recommended_courses.append({'name': course_name, 'url': course_url})
        
    return recommended_courses
        

In [111]:
df.head()

,Course Name,Course URL,tags
0,Write A Feature Length Screenplay For Film Or ...,https://www.coursera.org/learn/write-a-feature...,"write,a,feature,length,screenplay,for,film,or,..."
1,Business Strategy Business Model Canvas Analys...,https://www.coursera.org/learn/canvas-analysis...,"business,strategy,business,model,canvas,analys..."
2,Silicon Thin Film Solar Cells,https://www.coursera.org/learn/silicon-thin-fi...,"silicon,thin,film,solar,cellsadvancedthis,cour..."
3,Finance for Managers,https://www.coursera.org/learn/operational-fin...,"finance,for,managersintermediatewhen,it,comes,..."
4,Retrieve Data using Single-Table SQL Queries,https://www.coursera.org/learn/single-table-sq...,"retrieve,data,using,single-table,sql,queriesbe..."


In [112]:
recommend("Retrieve Data using Single-Table SQL Queries")

[{'name': 'Creating Database Tables with SQL',
  'url': 'https://www.coursera.org/learn/creating-database-tables-with-sql'},
 {'name': 'Manipulating Data with SQL',
  'url': 'https://www.coursera.org/learn/manipulating-data-with-sql'},
 {'name': 'Create Relational Database Tables Using SQLiteStudio',
  'url': 'https://www.coursera.org/learn/create-relational-database-table-sqlitestudio'},
 {'name': 'Retrieve Data with Multiple-Table SQL Queries',
  'url': 'https://www.coursera.org/learn/multiple-table-sql-queries'},
 {'name': 'Advanced SQL Retrieval Queries in SQLiteStudio',
  'url': 'https://www.coursera.org/learn/Advanced-sql-retrieval-queries-in-sqlitestudio'},
 {'name': 'Relational Database Support for Data Warehouses',
  'url': 'https://www.coursera.org/learn/dwrelational'}]

In [113]:
pickle.dump(similarity, open("models/similarity.pkl", "wb"))
pickle.dump(df[["Course Name", "Course URL"]].to_dict('records'), open("models/course_list.pkl", "wb"))
pickle.dump(df[["Course Name", "Course URL"]], open("models/course_list.pkl", "wb"))